[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/18_LogisticReg.ipynb#copy=true)

# MATH 232 - Math Models w/ Tech
## PyTorch & Logistic Regression
### Instructor: Prof. Mario Bañuelos 

In [1]:
# Import necessary packages
from IPython.display import HTML, Image
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from skimage import data
from sklearn import datasets
import seaborn as sns; sns.set()
import torch
import torch.nn as nn
import torch.nn.functional as F

## Outline
 - [Logistic Regression](#log)
     - [$P(Y=1)$](#prob)
     - [Loss Functions](#loss)
 - [Building a Model](#build)

***

## Logistic Regression  <a id='log'> </a>

In statistics, <b> logistic regression </b> is a model to calculate the probability of a certain class (e.g., type of flower, win/lose, pass/fail). We will motivate this discussion with a two class ($Y=0$ or $Y=1$) problem with two predictors, $x_1$ and $x_2$.

<a id='prob'> </a>
Let $p = P(Y=1)$. If we assume a linear relationship between the predictors and the log-odds, $\ell$, then we can write this down as 

$$
\ell = \log \left( \frac{p}{1-p} \right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2,
$$

<b> how would we think about solving for $p$? </b>
***

**Practice!** Solve for $p$ -- and write your solution into this notebook as $\alpha = \beta_0 + \beta_1 x_1 + \beta_2 x_2$,


$$
p = \frac{e^{\alpha}}{1 + e^{\alpha}}
$$

This is called the **sigmoid** function.

### Loss (or Cost) Functions  <a id='loss'> </a>

**Binary Cross Entropy Loss**

Now that we have a way to calculate $p$, we still need to find a way to update the parameters $\beta$s. We want to minimize the error (and correctly classify the observations), so we introduce the following loss (sometimes called cost) function

$$
\mathcal{L}(p, Y) = -Y \log(p) - (1-Y)\log(1-p)
$$

<b> Practice! </b> With a partner, 
   - Create a vector of 1000 probabilities (call it `pvec`) (between 0 and 1) 
   - Create the loss function above in Python 
   - Run the function on the probabilities with $Y=1$ (i.e. `myloss(pvec, 1)`)
   - Plot the loss function ($p$ vs. $\mathcal{L}$).
   - Add labels and interpret what you see!

It turns out, we want to solve the following minimization problem, 
$$
    \underset{\beta_0, \beta_1, \beta_2 \in \mathbb{R}}{\min} \mathcal{L}(p,Y).
$$

There are many ways to do so but one way is to take derivatives of $\mathcal{L}$ with respect to each parameter $\beta$, setting it equal to zero, and updating our $\beta$ estimates. <b> You have done this before! </b>

# Building a Model <a id='build'></a>

In [2]:
# generating fake data
# Here we generate some fake data
def lin(a,b,x): return a*x+b

def gen_fake(n, a, b):
    x1 = np.random.uniform(0,1,n) 
    x2 = np.random.uniform(0,1,n)
    noise = np.random.normal(1,3,n)
    y = np.zeros(n)
    #for i in range(n):
        #y.append(a*x1[i] + b*x2[i] + c + 0.1 * (-1)**i * noise[i])
    for i in range(n):
        if a*x1[i]+b>x2[i]:
            y[i] = 1
    return x1, x2, y

In [3]:
x1, x2, y = gen_fake(1000, 1, 0.1)

In [4]:
#y

In [5]:
#t = np.arange(-20, 20, 0.2)
#import matplotlib.pyplot as plt
#plt.scatter(x[:,0],x[:,1],c=y, s=8);
#
#plt.xlabel("x1"); plt.ylabel("x2");
#plt.plot(t, t + 0.5, 'r--')

In [9]:
# convert features and output into tensors
x = np.vstack([x1,x2]).T
x.shape
x

array([[0.13914298, 0.34656092],
       [0.6401584 , 0.30992174],
       [0.49230615, 0.6253749 ],
       ...,
       [0.35547335, 0.16776579],
       [0.26450839, 0.09922044],
       [0.91456623, 0.31920433]])

In [10]:
x = torch.tensor(x).float()
y = torch.tensor(y).float()

/Users/mariob/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [11]:
model = torch.nn.Sequential(
    torch.nn.Linear(2, 1),
)
model

Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
)

In [12]:
print([p for p in model.parameters()])

[Parameter containing:
tensor([[ 0.6657, -0.5162]], requires_grad=True), Parameter containing:
tensor([-0.6708], requires_grad=True)]


In [13]:
model(x).shape

torch.Size([1000, 1])

In [21]:
x1, x2, y = gen_fake(10000, 1., 0.5)
x = np.vstack([x1,x2]).T
x = torch.tensor(x).float()
y = torch.tensor(y).float()

In [32]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = x1, y=x2, mode='markers', marker=dict(color=y)))
fig.show()

In [22]:
learning_rate = 0.1
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [23]:
model(x)

tensor([[60.5001],
        [ 5.7973],
        [ 6.0672],
        ...,
        [48.3680],
        [34.4108],
        [46.0440]], grad_fn=<AddmmBackward>)

In [24]:
for t in range(10000):
    # Forward pass: compute predicted y using operations on Variables
    # y_hat is our prediction (based on x)
    y_hat = model(x)
    # define our loss function
    loss = F.binary_cross_entropy(torch.sigmoid(y_hat), y.unsqueeze(1))
    if t % 1000 == 0: print(loss.item())
       
    # Before the backward pass, use the optimizer object to zero all of the
    # gradients for the variables
    optimizer.zero_grad()
    loss.backward()
    
    # Calling the step function on an Optimizer makes an update to its
    # parameters
    optimizer.step()

0.007925423793494701
0.005361112300306559
0.004018217790871859
0.0032250864896923304
0.002680593403056264
0.0022854406852275133
0.001998726511374116
0.0017808435950428247
0.0016099746571853757
0.0014730131952092052


In [25]:
print([p for p in model.parameters()])

[Parameter containing:
tensor([[ 610.3480, -609.7010]], requires_grad=True), Parameter containing:
tensor([304.7502], requires_grad=True)]


* other params 
* 649, -650, 325 (a=1, b=0.5)
* 146, -145, 13 (a=1, b=0.1)
* -0.5111,  0.6620, 0.6055

$$ x_1 - x_2 + 0.5 = 0$$

In [34]:
model2 = torch.nn.Sequential(
    torch.nn.Linear(2, 2),
    torch.nn.Sigmoid(),
    torch.nn.Linear(2, 3),
    torch.nn.Sigmoid(),
    torch.nn.Linear(3, 1)
)
model2

Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=2, out_features=3, bias=True)
  (3): Sigmoid()
  (4): Linear(in_features=3, out_features=1, bias=True)
)